# 1. Know the AI estate

**Question:** What AI do we have, where does it run, and who owns it?

This notebook uses the repository's typed evidence model. The default dataset is **synthetic**, not customer inventory. Azure Commercial demonstrations do not establish GCC or Azure Government feature parity. A resource, an agent definition, a published application, and an Entra identity are different objects.

The optional live collector reads one configured Commercial resource group and Foundry project. It does not deploy resources, grant roles, send prompts, or enumerate the entire organization.

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/governance').is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
from IPython.display import Markdown, display
from src.governance.evidence import synthetic_estate, overview, coverage, markdown_table

estate = synthetic_estate()
display(Markdown('**Mode: synthetic / offline. No Azure calls.**'))
display(Markdown(markdown_table(overview(estate))))

## Inventory is not visibility

Every asset carries a source, cloud, owner, evidence kind, and UTC collection time. Every observation references an asset. Unknown owners remain **Unassigned**. Missing connectors remain gaps rather than a zero-risk score.

Synthetic timestamps belong to the fixture; they are not refreshed to make the dataset appear live. Snapshots preserve original collection times and relabel live observations as snapshots.

In [ ]:
print('Evidence coverage:', coverage(estate))
assert all(asset.evidence_kind == 'synthetic' for asset in estate.assets)
assert any(row.status == 'unknown' for row in estate.evidence)
print('Unknown is a finding about coverage, not a count of security risks.')

## Optional scoped live reads

`LIVE_READS` is false by default. The collector validates the active Azure CLI cloud/subscription and an optional expected tenant before any service reads. `AZURE_AI_PROJECT_RESOURCE_ID` supplies the target scope; expected context can be constrained with `EXPECTED_AZURE_SUBSCRIPTION_ID` and `EXPECTED_AZURE_TENANT_ID`. Graph permissions are separate from Azure resource permissions.

The collector reads resource metadata, agent definitions, Agent 365 configuration, aggregate gateway logs where a single workspace is present, and account-matched active Defender alert metadata. Agent definitions do not prove publication; source alerts do not establish agent-risk attribution. Permission failures are explicit unavailable/error rows. A failed top-level scope or inventory read raises an error. Agent 365 risk feeds are not implemented.

In [ ]:
LIVE_READS = False

if LIVE_READS:
    from dotenv import load_dotenv
    from src.governance.live import collect_live
    load_dotenv(ROOT / '.env')
    # Object IDs are optional inputs; omitted hosted coverage stays unknown.
    live_estate = collect_live(hosted_principal_ids=())
    display(Markdown(markdown_table(overview(live_estate))))
    display(Markdown(markdown_table([row.model_dump(mode='json') for row in live_estate.evidence])))
else:
    print('Live reads disabled. Synthetic inventory remains clearly labelled.')